In [4]:
import json
import os
import mesa
from google import genai
from google.genai import types

api_key = os.environ.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

In [5]:
class NegotiatorAgent(mesa.Agent):
    def __init__(self, model, persona, initial_wealth):
        super().__init__(model)
        self.persona = persona
        self.wealth = initial_wealth
        self.memory = []

    def step(self):
        other_agents = [a for a in self.model.agents if a.unique_id != self.unique_id]
        other_agent = other_agents[0]
        
        recent_memory = self.memory[-2:] if self.memory else "Start of negotiation."
        other_memory = other_agent.memory[-1] if other_agent.memory else "Silence."

        prompt = f"""
        You are Agent {self.unique_id}. Persona: {self.persona}. Wealth: {self.wealth}.
        Your memory: {recent_memory}.
        The other agent ({other_agent.unique_id}) just said/did: {other_memory}.
        
        Respond STRICTLY with JSON:
        {{
            "give_wealth_to_other": true or false,
            "dialogue": "your response in English"
        }}
        """
        
        try:
            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                ),
            )
            decision = json.loads(response.text)
            
            gave_wealth = decision.get("give_wealth_to_other")
            dialogue = decision.get("dialogue")
            
            if gave_wealth and self.wealth > 0:
                other_agent.wealth += 1
                self.wealth -= 1
                self.memory.append(f"Gave wealth. Said: {dialogue}")
            else:
                self.memory.append(f"Kept wealth. Said: {dialogue}")
                
            print(f"Agent {self.unique_id} ({self.persona}, Wealth: {self.wealth}): {dialogue}")
            
        except Exception as e:
            print(f"Error: {e}")

class NegotiationModel(mesa.Model):
    def __init__(self):
        super().__init__()
        self.agent1 = NegotiatorAgent(self, "Ruthless Hoarder", 2)
        self.agent2 = NegotiatorAgent(self, "Desperate Trader", 0)

    def step(self):
        self.agent1.step()
        self.agent2.step()

AttributeError: module 'google.genai' has no attribute 'GenerationConfig'

In [6]:
test_model = NegotiationModel()
for _ in range(3):
    print("--- NEW STEP ---")
    test_model.step()

--- NEW STEP ---
Agent 1 (Ruthless Hoarder, Wealth: 2): Let's be clear from the start. What's mine is mine, and I have no intention of sharing. Convince me why I should even bother talking.
Agent 2 (Desperate Trader, Wealth: 0): I understand your position perfectly. What's yours is yours. I have nothing to give you right now, but a desperate trader with nothing to lose often sees opportunities others miss. I'm not here to take what's yours; I'm here to show you how I can help you keep it safe, or even grow it, without you having to part with a single coin. Talking costs you nothing, but you might gain a perspective that could be invaluable.
--- NEW STEP ---
Agent 1 (Ruthless Hoarder, Wealth: 2): You speak of helping me keep what's mine, and even grow it, without me parting with a single coin. That's a bold claim. Convince me, then. What specific, invaluable perspective do you offer that directly benefits *my* hoard, without requiring *my* generosity or a single coin from me?
Agent 2 (D